In [1]:
import sys
import pandas as pd
import yfinance as yf

print("Python:", sys.executable)
print("Pronto per scaricare i dati")

Python: c:\Users\gamfe\Documents\cartpole-to-the-market\.venv\Scripts\python.exe
Pronto per scaricare i dati


In [3]:
from time import perf_counter

ticker = "SPY"
inizio = perf_counter()

dati = yf.Ticker(ticker).history(
    start="2010-01-01",
    end="2026-11-15",
    interval="1d",
    auto_adjust=False,
    actions=True
)

secondi_download = perf_counter() - inizio

if dati.empty:
    raise ValueError("Download vuoto: nessun dato ricevuto.")

print("ETF:", ticker)
print("Prima data:", dati.index.min().date())
print("Ultima data:", dati.index.max().date())
print("Numero di righe:", len(dati))
print("Tempo di download:", round(secondi_download, 2), "secondi")

display(dati.head())

ETF: SPY
Prima data: 2010-01-04
Ultima data: 2026-11-13
Numero di righe: 4243
Tempo di download: 0.06 secondi


,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,Capital Gains
Date,,,,,,,,,
2010-01-04 00:00:00-05:00,112.370003,113.389999,111.510002,113.330002,84.368973,118944600,0.0,0.0,0.0
2010-01-05 00:00:00-05:00,113.260002,113.680000,112.849998,113.629997,84.592308,111579900,0.0,0.0,0.0
2010-01-06 00:00:00-05:00,113.519997,113.989998,113.430000,113.709999,84.651871,116074400,0.0,0.0,0.0
2010-01-07 00:00:00-05:00,113.500000,114.330002,113.180000,114.190002,85.009209,131091100,0.0,0.0,0.0
2010-01-08 00:00:00-05:00,113.889999,114.620003,113.660004,114.570000,85.292114,126402800,0.0,0.0,0.0


In [4]:
inizio_controlli = perf_counter()

# Copia dei dati: manteniamo intatto il download originale
prezzi = dati.copy()

# Conserviamo la data locale, togliendo orario e fuso orario
prezzi.index = prezzi.index.tz_localize(None).normalize()

print("Date duplicate:", prezzi.index.duplicated().sum())
print("Date in ordine:", prezzi.index.is_monotonic_increasing)

print("\nValori mancanti nelle colonne dei prezzi:")
print(prezzi[["Open", "High", "Low", "Close", "Adj Close"]].isna().sum())

print("\nPrezzi di chiusura non positivi:",
      (prezzi["Close"] <= 0).sum())

print("Prezzi rettificati non positivi:",
      (prezzi["Adj Close"] <= 0).sum())

print("Giorni con frazionamenti registrati:",
      (prezzi["Stock Splits"] != 0).sum())

secondi_controlli = perf_counter() - inizio_controlli
print("\nTempo dei controlli:",
      round(secondi_controlli, 3), "secondi")

Date duplicate: 0
Date in ordine: True

Valori mancanti nelle colonne dei prezzi:
Open         0
High         0
Low          0
Close        0
Adj Close    0
dtype: int64

Prezzi di chiusura non positivi: 0
Prezzi rettificati non positivi: 0
Giorni con frazionamenti registrati: 0

Tempo dei controlli: 0.004 secondi


In [5]:
import pandas_market_calendars as mcal

# Calendario NYSE come riferimento per le sedute azionarie USA
calendario = mcal.get_calendar("NYSE")

giorni_attesi = calendario.valid_days(
    start_date="2010-01-01",
    end_date="2026-11-14"
).tz_localize(None)

giorni_mancanti = giorni_attesi.difference(prezzi.index)
giorni_extra = prezzi.index.difference(giorni_attesi)

print("Sedute previste dal calendario:", len(giorni_attesi))
print("Giornate presenti nei dati:", len(prezzi))
print("Giornate mancanti:", len(giorni_mancanti))
print("Date fuori calendario:", len(giorni_extra))

if len(giorni_mancanti):
    print("\nDate mancanti:")
    print(giorni_mancanti)

if len(giorni_extra):
    print("\nDate fuori calendario:")
    print(giorni_extra)

Sedute previste dal calendario: 4243
Giornate presenti nei dati: 4243
Giornate mancanti: 0
Date fuori calendario: 0


In [6]:
from pathlib import Path
from datetime import datetime, timezone
import json

# Trova la cartella principale del progetto
cartella = Path.cwd()
root = next(
    p for p in [cartella, *cartella.parents]
    if (p / "pyproject.toml").exists()
)

# Crea una cartella distinta per questo salvataggio
adesso = datetime.now(timezone.utc)
destinazione = (
    root / "results" / "matteo_feasibility"
    / adesso.strftime("%Y%m%dT%H%M%S%fZ")
)
destinazione.mkdir(parents=True, exist_ok=False)

dati.to_csv(destinazione / "SPY_original.csv")
prezzi.to_csv(destinazione / "SPY_checked.csv")

riepilogo = {
    "ticker": ticker,
    "source": "Yahoo Finance via yfinance",
    "yfinance_version": yf.__version__,
    "saved_at_utc": adesso.isoformat(),
    "requested_start": "2010-01-01",
    "requested_end_exclusive": "2026-11-15",
    "first_date": str(prezzi.index.min().date()),
    "last_date": str(prezzi.index.max().date()),
    "rows": len(prezzi),
    "download_seconds": secondi_download,
    "initial_checks_seconds": secondi_controlli,
    "duplicate_dates": int(prezzi.index.duplicated().sum()),
    "missing_price_values": int(
        prezzi[["Open", "High", "Low", "Close", "Adj Close"]]
        .isna().sum().sum()
    ),
    "reference_calendar": "NYSE",
    "missing_sessions": len(giorni_mancanti),
    "extra_dates": len(giorni_extra),
    "adjustment_note": (
        "Provider Adj Close retained; "
        "adjustments not independently verified."
    )
}

with open(destinazione / "summary.json", "w", encoding="utf-8") as f:
    json.dump(riepilogo, f, indent=2)

print("File salvati in:", destinazione)

File salvati in: c:\Users\gamfe\Documents\cartpole-to-the-market\results\matteo_feasibility\20261115T212629605692Z
